# README

Author: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助）

Created At: 2026-11-03

Last Modified At: 2026-11-03

## What does this file do?

補查1,502名議員登記候選人的來源明載性別，保留717筆既有值，補785筆。只處理性別，不以姓名、照片推測；不更動登記樣本與資格判斷。

## What does this file take?

[固定輸入清單](sex_focus_inputs_20261103.json)、2026登記名冊、三份更新前快照、2022中選會資料、2026維基現存及9月20日版本、媒體登記名單、號次對照。每份輸入SHA-256固定核對。

## What does this file output?

更新candidate_fields_current、candidate_master_current、council_background_20261019；另存council_sex_current、來源觀察、6筆來源差異紀錄及verification JSON。詳見[整理後資料文件](../processed/sex_focus_20261103_documentation.ipynb)。

## 比對規則與限制

縣市＋選區＋唯一中文姓名比對；NFKC、台／臺、温／溫正規化。原住民羅馬字拼法差異保留來源原文。11筆不同字姓名另以同選區、同號次逐筆核對，清單見[人工對照](sex_focus_name_review_20261103.json)；號次資料庫可能共用維基來源，不能當作性別獨立佐證。現存頁缺者使用固定舊版本，不刪除登記候選人。

2026次級來源不等於官方查核。張光儀保留2022官方男，另記2026維基女的未決衝突，禁止當作已核定2026值。陳泱瑚保留既有官方男；同號次維基「陳映瑚／女」另列異常。其餘4筆為維基前後版本性別不同。王珍玨的媒體描述只作佐證，不冒稱官方性別欄。

In [1]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT/'data/code'))
def sha(path): return hashlib.sha256((ROOT/path).read_bytes()).hexdigest()
def read(path): return pd.read_csv(ROOT/path,dtype=str,keep_default_na=False)
assert sha('data/code/sex_focus_inputs_20261103.json') == '80beae0e6bf829e6f4b1d481f122f31a64964c13ad1b904063198da2dca875a0'
from sex_focus_pipeline_20261103 import run
verification=run(ROOT)
verification

{'council_candidates': 1502,
 'previously_populated': 717,
 'newly_populated': 785,
 'populated': 1502,
 'current_source_conflict_unresolved': 1,
 'source_discrepancy_cases': 6,
 'source_tiers': {'secondary_2026_explicit_sex': 781,
  'prior_record_retained': 717,
  'official_2022_historical_current_conflict': 1,
  'media_2026_registration_explicit_sex': 3},
 'outputs': {'data/processed/candidate_fields_current.csv': {'rows': 31660,
   'columns': 11,
   'sha256': '74ef1f3a77d24f73a5f3682682b18c97ca55843e7879ada7c1cc86b57dd0730c'},
  'data/processed/candidate_master_current.csv': {'rows': 1583,
   'columns': 34,
   'sha256': 'f4806b229118b4f67383e4ed081144bffc0b81404600ffce646df6017c23eea4'},
  'data/processed/council_background_20261019.csv': {'rows': 1502,
   'columns': 53,
   'sha256': '32aebc40e4579333fa6f4d9476a763e1974537f6349f82f49396a88f7c58b33b'},
  'data/processed/council_sex_current.csv': {'rows': 1502,
   'columns': 18,
   'sha256': '21518248cbedff13985bd7775e705b1dbee59abc34

In [2]:
sex=read('data/processed/council_sex_current.csv')
assert sex.candidate_key.is_unique and len(sex)==1502
display(sex.groupby(['source_tier','sex']).size().rename('count').reset_index())
display(read('data/processed/council_sex_conflicts_20261103.csv'))

,source_tier,sex,count
0,media_2026_registration_explicit_sex,女,1
1,media_2026_registration_explicit_sex,男,2
2,official_2022_historical_current_conflict,男,1
3,prior_record_retained,女,285
4,prior_record_retained,男,432
5,secondary_2026_explicit_sex,女,252
6,secondary_2026_explicit_sex,男,529


,candidate_key,candidate_name,issue,source_a,value_a,source_b,value_b,resolution
0,2026|縣市議員|雲林縣第1選舉區|張光儀,張光儀,official_2022_vs_secondary_2026_unresolved,data/raw/cec_all_offices/106_elcand.csv:388,男,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,女,僅收錄2022官方申報值；2026衝突未裁定，exclude_from_unqualifie...
1,2026|縣市議員|嘉義縣第2選舉區|魏彰儀,魏彰儀,wiki_revision_disagreement,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,男,https://zh.wikipedia.org/w/index.php?oldid=944...,女,保留既有值或最新明載值；歷史版本差異另列，不視為獨立查證
2,2026|縣市議員|澎湖縣第4選舉區|李家萱,李家萱,wiki_revision_disagreement,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,女,https://zh.wikipedia.org/w/index.php?oldid=944...,男,保留既有值或最新明載值；歷史版本差異另列，不視為獨立查證
3,2026|縣市議員|金門縣第1選舉區|陳泱瑚,陳泱瑚,secondary_name_and_sex_disagree_with_prior_off...,https://db.cec.gov.tw/ | data/raw/cec_all_offi...,男,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,女,保留既有官方歷史值；維基同號次姓名及性別均不同，不覆寫；需當屆官方複核
4,2026|縣市議員|基隆市第1選舉區|王珍玨,王珍玨,wiki_revision_disagreement,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,女,https://zh.wikipedia.org/w/index.php?oldid=944...,男,保留既有值或最新明載值；歷史版本差異另列，不視為獨立查證
5,2026|縣市議員|基隆市第2選舉區|陳宜,陳宜,wiki_revision_disagreement,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,女,https://zh.wikipedia.org/w/index.php?oldid=944...,男,保留既有值或最新明載值；歷史版本差異另列，不視為獨立查證
